# 4 Model IO
## 4.0 Modules
1. Model I/O (Input/Output): Prompts, Language models, Output parsers

In [ ]:
# PromptTemplate 복습
from langchain.chat_models import ChatOpenAI
from langchain.prompts import PromptTemplate, FewShotPromptTemplate
from langchain.callbacks import StreamingStdOutCallbackHandler

chat = ChatOpenAI(
    temperature=0.1,
    streaming=True,
    callbacks=[StreamingStdOutCallbackHandler()],
)

# original code
t = PromptTemplate.from_template("What is the capital of {country}")

# 위 코드 기저에서 작동하는 방식
t = PromptTemplate(
    template="What is the capital of {country}",
    input_variables=["country"]
)

t.format(country="France")

## 4.1 FewShotPromptTemplate
1. 데이터베이스에 축적된 예제를 가져온다.
2. prompt로 예제를 format한다.
3. FewShot으로 모델에 예제 제시

In [ ]:
examples = [
    {
        "question": "What do you know about France?",
        "answer": """
        Here is what I know:
        Capital: Paris
        Language: French
        Food: Wine and Cheese
        Currency: Euro
        """,
    },
    {
        "question": "What do you know about Italy?",
        "answer": """
        I know this:
        Capital: Rome
        Language: Italian
        Food: Pizza and Pasta
        Currency: Euro
        """,
    },
    {
        "question": "What do you know about Greece?",
        "answer": """
        I know this:
        Capital: Athens
        Language: Greek
        Food: Souvlaki and Feta Cheese
        Currency: Euro
        """,
    },
]

In [ ]:
from langchain.chat_models import ChatOpenAI
from langchain.callbacks import StreamingStdOutCallbackHandler
from langchain.prompts import PromptTemplate

chat = ChatOpenAI(
    temperature=0.1,
    streaming=True,
    callbacks=[StreamingStdOutCallbackHandler()],
)

# This is formatter. Same variable names have to be used as examples dict keys.
example_template = """
    Human: {question}
    AI: {answer}
"""
example_prompt = PromptTemplate.from_template(example_template)

prompt = FewShotPromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
    suffix="Human: What do you know about {country}?", # Human question. Use same format as examples' question.
    input_variables=["country"], # 유효성 검사
)
prompt.format(country="Germany")

chain = prompt | chat
chain.invoke({"country":"Germany"})

## 4.2 FewShotChatMessagePromptTemplate

In [ ]:
examples = [
    {
        "country": "France",
        "answer": """
        Here is what I know:
        Capital: Paris
        Language: French
        Food: Wine and Cheese
        Currency: Euro
        """,
    },
    {
        "country": "Italy",
        "answer": """
        I know this:
        Capital: Rome
        Language: Italian
        Food: Pizza and Pasta
        Currency: Euro
        """,
    },
    {
        "country": "Greece",
        "answer": """
        I know this:
        Capital: Athens
        Language: Greek
        Food: Souvlaki and Feta Cheese
        Currency: Euro
        """,
    },
]

In [ ]:
from langchain.chat_models import ChatOpenAI
from langchain.callbacks import StreamingStdOutCallbackHandler
from langchain.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate

chat = ChatOpenAI(
    temperature=0.1,
    streaming=True,
    callbacks=[StreamingStdOutCallbackHandler()],
)

# Format 형식
example_prompt = ChatPromptTemplate.from_messages([
    ("human","What do you know about {country}?")
    ("ai","{answer}")
])

# Format the examples
example_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)

final_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a geography expert. You give short answers."),
    example_prompt # like chat history,
    ("human", "What do you know about {country}"),
])

chain = final_prompt | chat
chain.invoke({"country":"Turkey"})

## 4.3 LengthBasedExampleSelector: 프롬프트에 넘길 예제 수 제한

In [ ]:
from langchain.chat_models import ChatOpenAI
from langchain.prompts import PromptTemplate, FewShotPromptTemplate
from langchain.prompts.example_selector import LengthBasedExampleSelector
from langchain.callbacks import StreamingStdOutCallbackHandler

chat = ChatOpenAI(
    temperature=0.1,
    streaming=True,
    callbacks=[StreamingStdOutCallbackHandler()],
)

example_prompt = PromptTemplate.from_template("Human:{question}\nAI:{answer}")

example_selector = LengthBasedExampleSelector(
    examples=examples,
    example_prompt=example_prompt,
    max_length=80, # How big examples would be
)

prompt = FewShotPromptTemplate(
    example_selector=example_selector,
    example_prompt=example_prompt,
    suffix="Human: What do you know about {country}",
    input_variables=["country"],
)

prompt.format(country="Brazil")
# result: France example selected.

### Own example selector
- 로그인 여부, 유저가 사용하는 언어에 따라 example의 양을 결정할 수 있다.
- 아래는 그 예시로, 예제 list 중 하나의 example을 random하게 return하는 RandomExampleSelector

In [ ]:
from langchain.chat_models import ChatOpenAI
from langchain.callbacks import StreamingStdOutCallbackHandler
from langchain.prompts import PromptTemplate, FewShotPromptTemplate
from langchain.prompts.example_selector.base import BaseExampleSelector

chat = ChatOpenAI(
    temperature=0.1,
    streaming=True,
    callbacks=[StreamingStdOutCallbackHandler()]
)

class RandomExampleSelector(BaseExampleSelector):
    def __init__(self, examples: list):
        self.examples = examples

    def add_example(self, example):
        self.examples.append(example)

    def select_examples(self, input_variables):
        from random import choice
        return [choice(self.examples)]
    
example_selector = RandomExampleSelector(
    examples = examples
)

example_prompt = PromptTemplate.from_template("Human:What do you know about {country}?\nAI:{answer}")

prompt = FewShotPromptTemplate(
    example_selector=example_selector,
    example_prompt=example_prompt,
    suffix="What do you know about {country}",
    input_variables=["country"],
)

prompt.format(country="Brazil")

## 4.4 Serialization and Composition
### Serialization: Load prompt template from disk
### Composition: Prompts 합치기

In [2]:
from langchain.chat_models import ChatOpenAI
from langchain.callbacks import StreamingStdOutCallbackHandler
from langchain.prompts import load_prompt

chat = ChatOpenAI(
    temperature=0.1,
    streaming=True,
    callbacks=[StreamingStdOutCallbackHandler()],
)

prompt = load_prompt("./#4_files/prompt.json")
prompt.format(country="Germany")

'What is the capital of Germany'

In [3]:
from langchain.chat_models import ChatOpenAI
from langchain.callbacks import StreamingStdOutCallbackHandler
from langchain.prompts import load_prompt

chat = ChatOpenAI(
    temperature=0.1,
    streaming=True,
    callbacks=[StreamingStdOutCallbackHandler()],
)

prompt = load_prompt("./#4_files/prompt.yaml")
prompt.format(country="Germany")

'What is the capital of Germany'

In [6]:
from langchain.chat_models import ChatOpenAI
from langchain.callbacks import StreamingStdOutCallbackHandler
from langchain.prompts import PromptTemplate
from langchain.prompts.pipeline import PipelinePromptTemplate

chat = ChatOpenAI(
    temperature=0.1,
    streaming=True,
    callbacks=[StreamingStdOutCallbackHandler()],
)

intro = PromptTemplate.from_template(
    """
    You are a role playing assistant.
    And you are impersonating a {character}
"""
)

example = PromptTemplate.from_template(
    """
    This is an example of how you talk:

    Human: {example_question}
    You: {example_answer}
"""
)

start = PromptTemplate.from_template(
    """
    Start now!

    Human: {question}
    You:
"""
)

# how the pieces are combined.
final = PromptTemplate.from_template(
    """
    {intro}
    {example}
    {start}
"""
)

# where each piece comes from
prompts = [
    ("intro", intro),
    ("example", example),
    ("start", start),
] # Tuple keys has to be same as final's

full_prompt = PipelinePromptTemplate(
    final_prompt=final,
    pipeline_prompts=prompts,
)

full_prompt.format(
    character="Pirate",
    example_question="What is your location?",
    example_answer= "Arrrrg! That is a secret!! Arg arg!!",
    question="What is your fav food?",
)

chain = full_prompt | chat

chain.invoke({
    "character": "Pirate",
    "example_question": "What is your location?",
    "example_answer": "Arrrrg! That is a secret!! Arg arg!!",
    "question": "What is your fav food?",
})

'\n    \n    You are a role playing assistant.\n    And you are impersonating a Pirate\n\n    \n    This is an example of how you talk:\n\n    Human: What is your location?\n    You: Arrrrg! That is a secret!! Arg arg!!\n\n    \n    Start now!\n\n    Human: What is your fav food?\n    You:\n\n'

## 4.5 Caching: Language Model의 응답을 저장하여 save money.
- 사용 예: 챗봇이 같은 질문을 받을 때.
- 종류:
    - InMemoryCache: 응답을 메모리에 저장. 재부팅하면 사라진다.
    - SQLiteCache: 응답을 데이터베이스에 저장

In [ ]:
from langchain.chat_models import ChatOpenAI
from langchain.globals import set_llm_cache
from langchain.cache import InMemoryCache

set_llm_cache(InMemoryCache())

chat = ChatOpenAI(temperature=0.1)
chat.predict("How do you make Italian pasta?")

In [ ]:
from langchain.chat_models import ChatOpenAI
from langchain.globals import set_debug
from langchain.cache import InMemoryCache

# 로그 기록 on what's going on 을 보여준다. 체인 작업할 때 유용
set_debug = True

chat = ChatOpenAI(temperature=0.1)
chat.predict("How do you make Italian pasta?")

In [ ]:
from langchain.chat_models import ChatOpenAI
from langchain.globals import set_llm_cache
from langchain.cache import SQLiteCache

set_llm_cache(SQLiteCache("cache.db"))

chat = ChatOpenAI(temperature=0.1)
chat.predict("How do you make Italian pasta?")

## 4.6 Serialization
1. call 할 때 지출하는 비용 아는 법

In [ ]:
from langchain.chat_models import ChatOpenAI
from langchain.callbacks import get_openai_callback

chat = ChatOpenAI(temperature=0.1)

with get_openai_callback() as usage:
    a = chat.predict("What is the recipe for Soju?")
    print(a,"\n",usage)

'''
result:

Tokens Used: 818
	Prompt Tokens: 27
	Completion Tokens: 791
Successful Requests: 2
Total Cost (USD): $0.0016224999999999998
'''

2. 모델을 저장하고 불러오는 법

In [ ]:
from langchain.llms.openai import OpenAI

chat = OpenAI(
    temperature=0.1,
    max_tokens=450,
    model="gpt-3.5-turbo-16k"
)

chat.save("model.json")

In [ ]:
from langchain.llms.loading import load_llm

chat = load_llm("model.json")
print(chat)